### Pre-Processing (Idea Specific)

In [ ]:
from shared.config import PROCESSED
import pandas as pd
import sys
sys.path.insert(0, '..')


CLEAN = PROCESSED / 'clean'
plays = pd.read_parquet(CLEAN / 'player_play.parquet')
games = pd.read_parquet(PROCESSED / 'parquet' /
                        'nfl-big-data-bowl-2027__games.parquet')
players = pd.read_parquet(PROCESSED / 'parquet' /
                          'nfl-big-data-bowl-2027__players.parquet')

df = (plays.merge(games[['game_id', 'season', 'season_type']], on='game_id')
           .merge(players[['nfl_id', 'nfl_position']], on='nfl_id'))

steps = {}
steps['all player-plays'] = len(df)
df = df[df['nfl_position'].isin(['T', 'G', 'C'])]
steps['OL by nfl_position'] = len(df)
df = df[df['pass_rushers_encountered'].notna() & df['route_ran'].isna()]
steps['pass blocking'] = len(df)
df = df[df['season_type'].eq('REG')]
steps['regular season only'] = len(df)
df = df[df['play_nullified_by_penalty'].ne('Y')]
steps['no nullified plays'] = len(df)
df = df[df['season'].ne(2025)]
steps['2025 held out'] = len(df)

df['short_dropback'] = df['dropback_duration'] <= 2
pd.Series(steps)

all player-plays       314197
OL by nfl_position     107720
pass blocking           60175
regular season only     48689
no nullified plays      48688
2025 held out           24428
dtype: int64

## Verify Short Dropback Ratio

In [5]:
df['short_dropback'].value_counts()

short_dropback
False    20174
True      4254
Name: count, dtype: int64

## Head to Validate Before Continuing

In [7]:
df[['nfl_id', 'game_id', 'play_id', 'season', 'nfl_position',
    'lined_up_position', 'dropback_duration', 'pressure_allowed']].head(100)

,nfl_id,game_id,play_id,season,nfl_position,lined_up_position,dropback_duration,pressure_allowed
8,55976,2023091801,56,2023,T,T,1.835,False
16,55973,2023101503,3590,2023,G,G,1.635,False
50,57224,2024120104,2658,2024,G,G,2.142,False
54,55874,2024091513,4028,2024,T,T,3.184,False
62,55922,2024102007,3851,2024,C,C,2.586,False
...,...,...,...,...,...,...,...,...
1347,55903,2024091600,3995,2024,G,G,2.479,False
1368,55870,2024101302,4103,2024,T,T,1.849,False
1372,55982,2023110506,3734,2023,G,G,2.436,False
1381,57172,2024092204,2642,2024,C,C,2.244,False


## Pressure Allowed Shared

In [ ]:
df['pressure_allowed'].astype(float).mean(
), df['pressure_allowed'].isna().mean()

(np.float64(0.08748157851645652), np.float64(0.0))

## Per Player Counts

In [9]:
per_player = df.groupby('nfl_id').size()
per_player.describe(), (per_player >= 100).sum()

(count      56.000000
 mean      436.214286
 std       400.851943
 min         1.000000
 25%        53.750000
 50%       378.000000
 75%       619.500000
 max      1289.000000
 dtype: float64,
 np.int64(40))

In [12]:
per_player.head(10)

len(per_player)

56